# 02 — EDA DENUE: Establecimientos Tech en la ZMG

**Proyecto:** GDL Ecosystem Intelligence  
**Fuente:** DENUE — INEGI API  
**Clasificación Tech:** SCIAN rama 51 (Información en medios masivos) + subconjunto por nombre de actividad  
**Municipios ZMG:** Guadalajara · Zapopan · Tlaquepaque · Tonalá · Tlajomulco · El Salto

---
**Objetivo:** Mapear la densidad de empresas tech por municipio y correlacionar con el crecimiento del nearshoring.

In [ ]:
# ── 0. Entorno ────────────────────────────────────────────────────────────────
import os
import sys
from pathlib import Path

project_root = Path(r'C:\Users\emmys\OneDrive\Documents\GDL-ECO-INT')
os.chdir(project_root)
if str(project_root / 'src') not in sys.path:
    sys.path.insert(0, str(project_root / 'src'))

from dotenv import load_dotenv
load_dotenv(project_root / '.env')
DENUE_TOKEN = os.getenv('DENUE_TOKEN')
assert DENUE_TOKEN, '❌ DENUE_TOKEN no encontrado en .env'
print(f'✅ Token DENUE cargado: {DENUE_TOKEN[:8]}...')

In [ ]:
# ── 1. Imports ────────────────────────────────────────────────────────────────
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import requests
import json
import time
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

# Intentar cargar geopandas (opcional — si no está instalado, skip mapas)
try:
    import geopandas as gpd
    from shapely.geometry import Point
    GEO_AVAILABLE = True
    print('✅ geopandas disponible — mapas geoespaciales activados')
except ImportError:
    GEO_AVAILABLE = False
    print('⚠️  geopandas no instalado — se usarán gráficos alternativos')
    print('   Para instalar: pip install geopandas')

plt.rcParams.update({
    'figure.dpi': 130,
    'figure.facecolor': '#FAFAFA',
    'axes.facecolor': '#FAFAFA',
    'axes.spines.top': False,
    'axes.spines.right': False,
})
PALETTE_TECH = '#1A73E8'
PALETTE_NEU  = '#5F6368'

print('✅ Librerías cargadas')

In [ ]:
# ── 2. Función de consulta DENUE API ─────────────────────────────────────────
BASE_URL = 'https://www.inegi.org.mx/app/api/denue/v1/consulta'

# Municipios ZMG: clave INEGI = estado(14) + municipio(3 dígitos)
ZMG_MUNICIPIOS = {
    '14039': 'Guadalajara',
    '14120': 'Zapopan',
    '14098': 'Tlaquepaque',
    '14101': 'Tonalá',
    '14097': 'Tlajomulco',
    '14070': 'El Salto',
}

# SCIAN relacionados con tech / nearshoring
# 51 = Información en medios masivos
# 54 = Servicios profesionales (incluye consultoría TI)
# Los filtramos más adelante por nombre de actividad
SCIAN_TECH = '51,517,5415,5416'


def consulta_denue_municipio(cve_municipio, actividad=SCIAN_TECH,
                              estrato='0', max_registros=1000,
                              token=DENUE_TOKEN):
    url = f'{BASE_URL}/Busca/{actividad}/jalisco/{cve_municipio}/0/{estrato}/{max_registros}/{token}'
    print(f'    URL: {url}')
    resp = requests.get(url, timeout=30)
    print(f'    Status: {resp.status_code}')
    print(f'    Response preview: {resp.text[:200]}')
    if resp.status_code != 200 or not resp.text.strip():
        return pd.DataFrame()
    try:
        return pd.DataFrame(resp.json())
    except Exception as e:
        print(f'    JSON error: {e}')
        return pd.DataFrame()

print('✅ Función DENUE configurada')

In [ ]:
# ── 3. Carga del CSV DENUE local ─────────────────────────────────────────────
RAW_CSV = Path('data/raw/DENUE/conjunto_de_datos/denue_inegi_14_.csv')

print(f'Leyendo: {RAW_CSV}')
df_raw = pd.read_csv(RAW_CSV, encoding='latin-1', dtype=str, low_memory=False)

print(f'\nColumnas: {df_raw.columns.tolist()}')
print(f'Shape   : {df_raw.shape}')

# Filtrar municipios ZMG por 'cve_mun'
ZMG_CLAVES = ['039', '120', '098', '101', '097', '070']
df_zmg = df_raw[df_raw['cve_mun'].isin(ZMG_CLAVES)].copy()
print(f'\nRegistros en ZMG: {len(df_zmg):,}')

# Verificar distribución del inicio de codigo_act antes de filtrar
print('\nDistribución primeros 2 dígitos de codigo_act (ZMG):')
print(df_zmg['codigo_act'].str[:2].value_counts().head(10).to_string())

# Filtrar sector tech: codigo_act empieza con '51'
df_denue = df_zmg[df_zmg['codigo_act'].str.startswith('51', na=False)].copy()
print(f'\nEstablecimientos tech (SCIAN 51): {len(df_denue):,}')

# Usar columna 'municipio' que contiene el nombre directamente
df_denue['municipio_nombre'] = df_denue['municipio']

# Conteo por municipio
print('\nEstablecimientos tech por municipio:')
print(df_denue.groupby('municipio_nombre').size().sort_values(ascending=False).to_string())

# Guardar resultado filtrado
out_path = Path('data/raw/DENUE/denue_zmg_tech.csv')
out_path.parent.mkdir(parents=True, exist_ok=True)
df_denue.to_csv(out_path, index=False, encoding='utf-8-sig')
print(f'\n✅ Guardado en: {out_path}')

In [ ]:
# ── 4. Limpieza y clasificación tech ─────────────────────────────────────────
# Mapear columnas reales de DENUE (ajustar si difieren)
# Columnas típicas DENUE: Nombre, Razon_social, Clase_actividad, Estrato,
#                         Longitud, Latitud, Municipio, Localidad

df = df_denue.copy()

# Detectar columna de actividad
col_actividad = next((c for c in df.columns if 'actividad' in c.lower() or 'clase' in c.lower()), None)
col_nombre    = next((c for c in df.columns if 'nombre' in c.lower() or 'razon' in c.lower()), None)
col_estrato   = next((c for c in df.columns if 'estrato' in c.lower()), None)
col_lat       = next((c for c in df.columns if 'latitud' in c.lower() or 'lat' in c.lower()), None)
col_lon       = next((c for c in df.columns if 'longitud' in c.lower() or 'lon' in c.lower()), None)

print(f'Columna actividad detectada : {col_actividad}')
print(f'Columna nombre detectada    : {col_nombre}')
print(f'Columna estrato detectada   : {col_estrato}')
print(f'Columna latitud detectada   : {col_lat}')
print(f'Columna longitud detectada  : {col_lon}')

# Keywords tech para filtrar si es necesario
KEYWORDS_TECH = [
    'software', 'desarrollo', 'tecnolog', 'informática', 'computación',
    'sistemas', 'cómputo', 'digital', 'datos', 'cloud', 'it ', ' ti ',
    'telecomunicac', 'internet', 'web', 'programación'
]

if col_nombre:
    patron = '|'.join(KEYWORDS_TECH)
    df['es_tech_kw'] = df[col_nombre].str.lower().str.contains(patron, na=False)
    print(f'\n✅ Establecimientos con keywords tech: {df["es_tech_kw"].sum():,}')

In [ ]:
# ── 5. VIZ 5: Establecimientos tech por municipio (barras) ───────────────────
conteo = (
    df.groupby('municipio_nombre')
    .size()
    .reset_index(name='n_establecimientos')
    .sort_values('n_establecimientos', ascending=True)
)

fig, ax = plt.subplots(figsize=(10, 5), facecolor='#FAFAFA')
bars = ax.barh(conteo['municipio_nombre'], conteo['n_establecimientos'],
               color=PALETTE_TECH, alpha=0.85, edgecolor='white', height=0.6)

for bar, val in zip(bars, conteo['n_establecimientos']):
    ax.text(val + conteo['n_establecimientos'].max() * 0.01,
            bar.get_y() + bar.get_height()/2,
            f'{val:,}', va='center', ha='left', fontsize=10, fontweight='bold')

ax.set_title('Establecimientos sector tech (DENUE)\npor municipio — ZMG', fontweight='bold', pad=15)
ax.set_xlabel('Número de establecimientos')
ax.set_ylabel('')

plt.tight_layout()
plt.savefig('reports/figures/05_establecimientos_por_municipio.png', bbox_inches='tight', dpi=150)
plt.show()
print('✅ Figura guardada → reports/figures/05_establecimientos_por_municipio.png')

In [ ]:
# ── 6. VIZ 6: Distribución por tamaño de empresa (per_ocu) ───────────────────
ESTRATO_MAP = {
    '0 a 5 personas':     'Micro (0-5)',
    '6 a 10 personas':    'Micro (6-10)',
    '11 a 30 personas':   'Pequeña (11-30)',
    '31 a 50 personas':   'Pequeña (31-50)',
    '51 a 100 personas':  'Mediana (51-100)',
    '101 a 250 personas': 'Mediana (101-250)',
    '251 y más personas': 'Grande (251+)',
}

df_denue['estrato_label'] = df_denue['per_ocu'].map(ESTRATO_MAP).fillna(df_denue['per_ocu'])

estrato_conteo = (
    df_denue.groupby(['municipio_nombre', 'estrato_label'])
    .size()
    .reset_index(name='n')
)

# Orden lógico de estratos de menor a mayor
orden_estratos = [
    'Micro (0-5)', 'Micro (6-10)',
    'Pequeña (11-30)', 'Pequeña (31-50)',
    'Mediana (51-100)', 'Mediana (101-250)',
    'Grande (251+)',
]

pivot_estrato = (
    estrato_conteo
    .pivot(index='municipio_nombre', columns='estrato_label', values='n')
    .fillna(0)
    .reindex(columns=[c for c in orden_estratos if c in estrato_conteo['estrato_label'].unique()])
)

fig, ax = plt.subplots(figsize=(12, 5), facecolor='#FAFAFA')
pivot_estrato.plot(kind='bar', ax=ax, colormap='Blues', alpha=0.9, edgecolor='white', stacked=True)

ax.set_title('Establecimientos tech por tamaño y municipio (DENUE)\nZMG — columna per_ocu', fontweight='bold')
ax.set_xlabel('')
ax.set_ylabel('Número de establecimientos')
ax.tick_params(axis='x', rotation=30)
ax.legend(title='Tamaño', bbox_to_anchor=(1.01, 1), loc='upper left', frameon=False, fontsize=8)

plt.tight_layout()
plt.savefig('reports/figures/06_establecimientos_por_estrato.png', bbox_inches='tight', dpi=150)
plt.show()
print('✅ Figura guardada → reports/figures/06_establecimientos_por_estrato.png')

In [ ]:
# ── 7. VIZ 7: Mapa de calor geoespacial (si geopandas disponible) ─────────────
if GEO_AVAILABLE and col_lat and col_lon:
    df_geo = df.copy()
    df_geo[col_lat] = pd.to_numeric(df_geo[col_lat], errors='coerce')
    df_geo[col_lon] = pd.to_numeric(df_geo[col_lon], errors='coerce')
    df_geo = df_geo.dropna(subset=[col_lat, col_lon])
    
    gdf = gpd.GeoDataFrame(
        df_geo,
        geometry=gpd.points_from_xy(df_geo[col_lon], df_geo[col_lat]),
        crs='EPSG:4326'
    )
    
    fig, ax = plt.subplots(figsize=(10, 10), facecolor='#F0F4F8')
    gdf.plot(ax=ax, color=PALETTE_TECH, alpha=0.4, markersize=8)
    
    ax.set_title('Distribución geoespacial — Establecimientos Tech ZMG\n(DENUE)', fontweight='bold')
    ax.set_xlabel('Longitud')
    ax.set_ylabel('Latitud')
    ax.annotate(f'n = {len(gdf):,} establecimientos', xy=(0.02, 0.02),
                xycoords='axes fraction', fontsize=10, color=PALETTE_NEU)
    
    plt.tight_layout()
    plt.savefig('reports/figures/07_mapa_establecimientos_tech.png', bbox_inches='tight', dpi=150)
    plt.show()
    print('✅ Mapa guardado → reports/figures/07_mapa_establecimientos_tech.png')

elif not GEO_AVAILABLE:
    print('ℹ️  Para activar el mapa: pip install geopandas')
else:
    print('ℹ️  Coordenadas no disponibles en los datos DENUE descargados')

In [ ]:
# ── 8. Exportar para cruce con ENOE y Power BI ────────────────────────────────
export_path = Path('data/processed/denue_tech_zmg.csv')
export_path.parent.mkdir(parents=True, exist_ok=True)

resumen_export = (
    df.groupby('municipio_nombre')
    .agg(
        n_establecimientos=('municipio_nombre', 'count'),
    )
    .reset_index()
)
df.to_csv(export_path, index=False, encoding='utf-8-sig')
print(f'✅ CSV exportado → {export_path}')

print('\n📌 HALLAZGOS — DENUE ZMG')
print('=' * 45)
print(resumen_export.to_string(index=False))
print(f'\n   Total establecimientos tech: {len(df):,}')

---
**Commit sugerido:**
```bash
git add notebooks/02_eda_denue.ipynb data/raw/DENUE/ data/processed/denue_tech_zmg.csv reports/figures/05*.png reports/figures/06*.png
git commit -m "analysis: EDA DENUE completo — establecimientos tech por municipio ZMG"
```